# 03 · Modelling & evaluation
**Report sections 4–5.** Baselines (naive, seasonal naive, moving average) vs. regularised linear models vs. tree ensembles,
evaluated with expanding-window time-series CV. Metric of record: **WAPE** (scale-free, robust to zero periods); also MAE/RMSE.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from superstore import config
from superstore.viz.plots import save_fig, set_style

set_style()
pd.set_option("display.max_columns", 50)

In [ ]:
from superstore.evaluation.cv import PanelTimeSeriesSplit
from superstore.features.build import build_panel
from superstore.models.train import run_cv, select_features, summarise
from superstore.viz.plots import plot_cv_results

orders = pd.read_parquet(config.ORDERS_FILE)
panel = build_panel(orders, freq="MS", group_cols=["market", "category"])
nums, cats = select_features(panel)
print(len(nums), "numeric:", nums)
print("categorical:", cats)

In [ ]:
cv = PanelTimeSeriesSplit(n_splits=4, horizon=3, min_train=13)
results = run_cv(panel, cv=cv)
summarise(results)

In [ ]:
ax = plot_cv_results(results, metric="wape")
save_fig(ax.figure, "cv_wape")

## Next steps
- Hyperparameter tuning *inside* the temporal folds (e.g. `GridSearchCV(cv=PanelTimeSeriesSplit(...))`)
- Profit as a second target; discount interaction model (statsmodels OLS with `discount * C(category)`) for inference
- SARIMA / ETS per series as a classical-statistics benchmark
- SHAP on the best tree model → which lags/seasonal features matter
- Residual analysis per market/category